# Entregable 1 — Ayuda automática desde Stack Overflow

Primero aparece el enunciado con sus ejemplos y después la solución y sus pruebas.
Ejecuta las celdas de arriba abajo con **Shift + Enter**. Necesitas conexión a Internet.

### Preparación
La siguiente celda instala las librerías en el Python del cuaderno. BeautifulSoup es gratuita.
`%pip` es una instrucción de Jupyter para instalar librerías; no forma parte de la función del ejercicio.
Después importamos las herramientas que vamos a utilizar.


In [ ]:
%pip install numpy requests beautifulsoup4


In [1]:
import numpy as np                 # Para los ejemplos del enunciado.
import requests                    # Para pedir información a Internet.
from bs4 import BeautifulSoup      # Para leer HTML y extraer su texto.
import webbrowser                  # Para abrir el enlace en el navegador.


## Web scraping exercise

Define a generic function `SOS_help` which retrieves help results from Stack Overflow Stunning results. <br>

The following command works just fine:

In [2]:
import numpy as np
np.random.uniform(-1, 1, size=100)

`print_out` prints command execution

In [3]:
def print_output(command):
    command = 'print(' + command + ')'
    return exec(command)

In [4]:
print_output(command = 'np.random.uniform(-1, 1, size=100)')

[ 0.35253641 -0.37786646 -0.63113121 -0.98062876  0.0701952   0.36048696
  0.46213233 -0.43578597 -0.50772689  0.51215888 -0.1210727   0.52915701
  0.30500484  0.32610695 -0.27592512  0.36396977  0.31792803 -0.11551259
  0.44192096  0.70068507  0.96958249 -0.74214374  0.63102942 -0.52655353
 -0.98681264  0.16469115  0.00380515 -0.35595038 -0.22351477 -0.45064568
  0.1517365   0.74542824  0.43222741  0.14608876  0.66781728  0.72997167
 -0.04180542  0.58999537  0.71034215 -0.46570036 -0.88432567  0.17585992
  0.93521788  0.16571331 -0.83540659 -0.25627157  0.00629694 -0.46272529
 -0.35153221 -0.40219806  0.47411341 -0.47544644 -0.73795822 -0.50194592
  0.8499907  -0.74445507  0.36318219 -0.66628705 -0.66013364  0.15816726
  0.85669033 -0.6354117  -0.13819131  0.53520862 -0.52510425  0.98580621
 -0.38877334 -0.50349247  0.08975301 -0.39304123  0.71314718  0.12609207
  0.74703266 -0.04659166  0.773242   -0.74674146  0.60758445 -0.00630458
  0.30907698  0.32631616  0.93188438  0.50370751 -0

if `print_out` results in an error, this is printed

In [5]:
# Error intencionado: se captura para que el cuaderno siga ejecutándose.
try:
    print_output(command = 'nn.random.uniform(-1, 1, size=100)')
except Exception as error:
    print(f"{type(error).__name__}: {error}")

NameError: name 'nn' is not defined


In [6]:
# Error intencionado: se captura para que el cuaderno siga ejecutándose.
try:
    print_output(command = 'np.random.uniform(-1, 1, siz=100)')
except Exception as error:
    print(f"{type(error).__name__}: {error}")

TypeError: uniform() got an unexpected keyword argument 'siz'


Create a function `get_SOS_help` which: <br>
    - Prints "works as intended" if no error. <br>
    - Prints the first link from stack overflow related to the error. As an example: <br>
        `print_output(command = 'np.random.uniform(-1, 1, siz=100)'`
        should retrieve the following link:
        https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size <br>
    - Prints the most voted help
    - Opens a new browser using the link

## Solución

`get_SOS_help(command)` recibe código escrito como texto y sigue estas instrucciones:

1. Ejecuta el código. Si funciona, imprime `works as intended` y termina.
2. Si hay un error, busca una pregunta relacionada y muestra el primer enlace.
3. Muestra la respuesta de esa pregunta con más votos.
4. Solicita abrir el enlace en una ventana del navegador.

### Herramientas utilizadas y relación con los apuntes

- **requests y BeautifulSoup:** aparecen en `Web_Scraping_1.ipynb` y `Web_Scraping_2.ipynb`.
- **API:** `Web_Scraping_2.ipynb` incluye el apartado «APIs as an alternative to web scraping».
- **JSON:** el cuaderno `INE.ipynb`, dentro de los archivos RAW, contiene ejemplos de `r.json()`.
- **try / except:** está en la plantilla original de este ejercicio y permite gestionar errores.
- **webbrowser:** usamos una llamada de la biblioteca estándar de Python para cumplir la apertura del enlace.
  Esta llamada concreta se añade para el ejercicio; no la he localizado como ejemplo de código en los apuntes revisados.

Durante las comprobaciones, Stack Overflow mostró verificaciones humanas y bloqueos de acceso.
Por eso consultamos su API oficial para buscar preguntas y obtener respuestas.
**Sí usamos BeautifulSoup:** la respuesta de la API contiene HTML y extraemos su texto con `get_text()`.
Este método combina API y lectura de HTML; no descarga las preguntas mediante scraping directo de la web.

### Cómo leer el código

- Un **diccionario** agrupa opciones con nombres, como `"site": "stackoverflow"`.
- `requests.get(..., params=...)` envía esas opciones a la API. `timeout=20` limita la espera de conexión y lectura.
- `.json()` convierte la respuesta en listas y diccionarios de Python.
- `"items"` contiene la lista de preguntas o respuestas; `[0]` obtiene la primera.
- `"sort": "relevance"` ordena por relación con la búsqueda; `"sort": "votes"` ordena por puntuación.
- `"order": "desc"` pone primero el resultado de mayor relevancia o puntuación.
- `"pagesize": 1` pide un resultado y `"filter": "withbody"` incluye el HTML de la respuesta.

Estas opciones son propias de Stack Exchange y se explican aquí para poder entender la adaptación.

Antes de buscar quitamos los fragmentos entre comillas simples. Por ejemplo, `split("'")` separa
el mensaje de `siz` en tres partes: el texto anterior, `siz` y el texto posterior.
El bucle recoge las posiciones **0, 2, 4...**, que son las partes exteriores a las comillas.
Así una errata concreta no impide encontrar preguntas sobre el mismo tipo de error.
La búsqueda es aproximada: hay que leer la respuesta para valorar si resuelve nuestro problema.


In [7]:
def get_SOS_help(command):
    # 1. Ejecutar el código recibido.
    try:
        exec(command)
        print("works as intended")
        return
    except Exception as error:
        print(type(error).__name__ + ":", error)

        # Quitar del mensaje las partes entre comillas, como 'siz'.
        partes = str(error).split("'")
        busqueda = ""
        for posicion in range(0, len(partes), 2):
            busqueda = busqueda + partes[posicion]

    api = "https://api.stackexchange.com/2.3"
    enlace = ""

    # Las consultas de Internet también pueden fallar.
    try:
        # 2. Buscar la primera pregunta relacionada con el error.
        parametros = {
            "q": busqueda,
            "tagged": "python",
            "site": "stackoverflow",
            "sort": "relevance",
            "order": "desc",
            "pagesize": 1
        }
        respuesta = requests.get(api + "/search/advanced", params=parametros, timeout=20)
        respuesta.raise_for_status()  # Comprobar si la petición tuvo un error HTTP.
        datos = respuesta.json()

        if "items" not in datos:
            print("La API no ha podido realizar la búsqueda.")
            return

        preguntas = datos["items"]
        if not preguntas:
            print("No se ha encontrado ninguna pregunta relacionada.")
            return

        pregunta = preguntas[0]
        enlace = pregunta["link"]
        print("\nPrimer enlace:", enlace)

        # 3. Obtener la respuesta con más votos de esa pregunta.
        identificador = str(pregunta["question_id"])
        url_respuestas = api + "/questions/" + identificador + "/answers"
        parametros = {
            "site": "stackoverflow",
            "sort": "votes",
            "order": "desc",
            "pagesize": 1,
            "filter": "withbody"
        }
        respuesta = requests.get(url_respuestas, params=parametros, timeout=20)
        respuesta.raise_for_status()
        datos = respuesta.json()

        if "items" not in datos:
            print("La API no ha podido obtener las respuestas.")
        else:
            respuestas = datos["items"]
            if respuestas:
                html = respuestas[0]["body"]
                soup = BeautifulSoup(html, "html.parser")
                print("\nRespuesta más votada:")
                print(soup.get_text("\n", strip=True))
            else:
                print("La pregunta todavía no tiene respuestas.")

    except requests.RequestException:
        print("No se pudo consultar Internet. Comprueba la conexión y vuelve a intentarlo.")
    except ValueError:
        print("La respuesta de Internet no tiene el formato JSON esperado.")

    # 4. Abrir la pregunta si hemos encontrado un enlace.
    if enlace:
        if not webbrowser.open_new(enlace):
            print("No se pudo abrir el navegador. Abre el enlace anterior manualmente.")


### Prueba 1 · Código correcto

Debe imprimir `works as intended`, sin buscar en Internet ni abrir el navegador.

In [8]:
command = 'np.random.uniform(-1, 1, size=100)'
get_SOS_help(command)

works as intended


### Prueba 2 — El ejemplo del enunciado

Escribimos `siz` en lugar de `size` a propósito. Debe mostrar el error, buscar una pregunta,
imprimir su respuesta más votada y solicitar la apertura del enlace en el navegador.

En las comprobaciones, el primer resultado fue el enlace del enunciado:
https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size

El orden de los resultados puede cambiar. Esa pregunta trata de argumentos incorrectos de `uniform`;
para nuestro código concreto la solución es escribir **`size` en lugar de `siz`**.


In [9]:
command = 'np.random.uniform(-1, 1, siz=100)'
get_SOS_help(command)

TypeError: uniform() got an unexpected keyword argument 'siz'

Primer enlace: https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size

Respuesta más votada:
We must use
random.uniform
while importing the related class with an alias (using 'as') or else just use
import numpy
while importing
An example for using alias is :
from numpy import random as np_random
Then utilize
np_random.uniform()
(figured it out and had to use this to solve the problem - note I ran it on
colab
)
[Comprobación automática: apertura del navegador simulada; al ejecutar tú la celda se solicitará abrirlo realmente.]


## Create a malfunctioning code and use this function on it

### Prueba 3 · Código propio que falla

Convertir a número un texto que no lo es (`int("abc")`) produce un `ValueError`.

In [10]:
command = 'int("abc")'
get_SOS_help(command)

ValueError: invalid literal for int() with base 10: 'abc'

Primer enlace: https://stackoverflow.com/questions/1841565/valueerror-invalid-literal-for-int-with-base-10

Respuesta más votada:
The error message means that the string provided to
int
could not be parsed as an integer. The part at the end, after the
:
, shows the string that was provided.
In the case described in the question, the input was an
empty string
, written as
''
.
Here is another example - a string that represents a floating-point value cannot be converted directly with
int
:
>>> int('55063.000000')
Traceback (most recent call last):
  File "<stdin>", line 1, in <module>
ValueError: invalid literal for int() with base 10: '55063.000000'
Instead, convert to
float
first:
>>> int(float('55063.000000'))
55063
[Comprobación automática: apertura del navegador simulada; al ejecutar tú la celda se solicitará abrirlo realmente.]


## Comprobación final

Para revisar el trabajo, ejecuta **Prueba 1**, **Prueba 2** y **Prueba 3**, y guarda el cuaderno.
La primera debe mostrar `works as intended`. Las otras dos deben buscar ayuda para sus errores.
Si no hay Internet o la API limita las consultas, aparecerá un mensaje explicativo.

`exec` ejecuta código: en estas pruebas usamos únicamente los ejemplos conocidos del ejercicio.
La configuración del navegador puede hacer que la apertura se realice en una pestaña.
Si Python muestra un problema de certificados, habrá que configurar los certificados del entorno;
no es necesario desactivar su verificación.

Referencias:
- [BeautifulSoup: biblioteca gratuita con licencia MIT](https://pypi.org/project/beautifulsoup4/)
- [Búsqueda en la API de Stack Exchange](https://api.stackexchange.com/docs/advanced-search)
- [Respuestas ordenadas por votos](https://api.stackexchange.com/docs/answers-on-questions)

**Verificación:** las tres pruebas se ejecutaron el 2 de noviembre de 2026. Las dos búsquedas obtuvieron respuestas reales. La apertura del navegador se simuló, como indican las salidas. En la comprobación se usaron los certificados de confianza de Windows. La instalación se omitió porque las dependencias ya estaban disponibles para el verificador.
